<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/HRP_Replication_LopezDePrado2016.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HRP Replication: López de Prado (2016)


## Experiment

Monte Carlo comparison of three portfolio construction methods:
- **HRP**: Hierarchical Risk Parity (paper's contribution)
- **CLA**: Critical Line Algorithm — minimum variance portfolio (via pypfopt CriticalLineAlgorithm, Bailey & López de Prado 2013)
- **IVP**: Inverse Variance Portfolio (traditional risk parity)

**Setup**:
- N = 10 assets: 5 uncorrelated base series + 5 correlated series
- sigma0 = 1% base volatility; correlated series noise = sigma0 * sigma1F
- Common and specific random shocks added to data each iteration
- 520 total observations, estimation window = 260, rebalancing every 22 obs
- 10,000 Monte Carlo iterations
- **Metric**: variance of terminal returns across MC iterations

---

## Notes on deviations from paper

- **CLA**: now uses `pypfopt.cla.CriticalLineAlgorithm` (Bailey & López de Prado 2013) directly. Long-only constraints (0 ≤ w ≤ 1, Σw = 1) match the paper exactly.
- **Python 3**: `xrange` → `range`; `pd.Series.append` → `pd.concat`.

## Notebook Map

- **Setup** -- imports, HRP/IVP/CLA portfolio functions, Monte Carlo data generator
- **Full Monte Carlo (10,000 iterations)** -- the main run, ~10-20 min
- **Interpretation** -- comparison against the paper's target results

## 1. Setup

- Installs `PyPortfolioOpt` (used for the CLA benchmark below)
- Imports: `numpy`/`pandas` for data handling, `scipy.cluster.hierarchy` for the single-linkage clustering HRP depends on, and `pypfopt.cla.CriticalLineAlgorithm` for the minimum-variance CLA benchmark

In [ ]:
!pip install pyportfolioopt -q

In [ ]:
import numpy as np
import pandas as pd
import random
import scipy.cluster.hierarchy as sch
from scipy.optimize import minimize
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pypfopt.cla import CLA as CriticalLineAlgorithm

## 2. Core HRP Functions

López de Prado's three-step HRP algorithm, implemented from Appendix A.3/A.4 with the minimal Python-3 adaptations noted in the code comments:
- `getQuasiDiag` runs single-linkage clustering on the correlation-based distance matrix, then reorders assets so similar ones sit next to each other (quasi-diagonalisation).
- `getRecBipart` recursively splits this ordered list in half and allocates capital between each half in inverse proportion to cluster variance, until every asset has a weight.
- `getIVP`/`getClusterVar` are the inverse-variance building blocks `getRecBipart` calls at each split, and `correlDist` converts the correlation matrix into the distance matrix clustering needs.

In [ ]:
# ============================================================
# Core HRP Functions
# Source: López de Prado (2016) Appendix A.3 / A.4
# Python 3 adaptations: xrange -> range; .append -> pd.concat
# ============================================================

def getIVP(cov, **kargs):
    """Inverse variance portfolio weights."""
    ivp = 1. / np.diag(cov)
    ivp /= ivp.sum()
    return ivp


def getClusterVar(cov, cItems):
    """Variance of a cluster using IVP weights."""
    cov_ = cov.loc[cItems, cItems]
    w_ = getIVP(cov_).reshape(-1, 1)
    cVar = np.dot(np.dot(w_.T, cov_), w_)[0, 0]
    return cVar


def getQuasiDiag(link):
    """Quasi-diagonalisation: sort clustered items by distance."""
    link = link.astype(int)
    sortIx = pd.Series([link[-1, 0], link[-1, 1]])
    numItems = link[-1, 3]
    while sortIx.max() >= numItems:
        sortIx.index = range(0, sortIx.shape[0] * 2, 2)
        df0 = sortIx[sortIx >= numItems]
        i = df0.index
        j = df0.values - numItems
        sortIx[i] = link[j, 0]
        df0 = pd.Series(link[j, 1], index=i + 1)
        sortIx = pd.concat([sortIx, df0])  # paper: sortIx.append(df0)
        sortIx = sortIx.sort_index()
        sortIx.index = range(sortIx.shape[0])
    return sortIx.tolist()


def getRecBipart(cov, sortIx):
    """Recursive bisection: compute HRP weights top-down."""
    w = pd.Series(1, index=sortIx)
    cItems = [sortIx]
    while len(cItems) > 0:
        cItems = [i[j:k] for i in cItems for j, k in
                  ((0, len(i) // 2), (len(i) // 2, len(i))) if len(i) > 1]
        for i in range(0, len(cItems), 2):  # paper: xrange
            cItems0 = cItems[i]
            cItems1 = cItems[i + 1]
            cVar0 = getClusterVar(cov, cItems0)
            cVar1 = getClusterVar(cov, cItems1)
            alpha = 1 - cVar0 / (cVar0 + cVar1)
            w[cItems0] *= alpha
            w[cItems1] *= 1 - alpha
    return w


def correlDist(corr):
    """Correlation-based distance: d_ij = sqrt(0.5*(1-rho_ij))."""
    dist = ((1 - corr) / 2.) ** .5
    return dist


def getHRP(cov, corr, **kargs):
    """
    Full HRP pipeline: cluster -> quasi-diag -> recursive bisect.
    Converts numpy arrays to DataFrames (A.4: cov/corr arrive as numpy arrays).
    """
    corr, cov = pd.DataFrame(corr), pd.DataFrame(cov)  # as in paper A.4
    dist = correlDist(corr)
    link = sch.linkage(dist, 'single')
    sortIx = getQuasiDiag(link)
    sortIx = corr.index[sortIx].tolist()  # recover labels
    hrp = getRecBipart(cov, sortIx)
    return hrp.sort_index()

## 3. CLA — Critical Line Algorithm Benchmark

- The paper's second comparison method: the minimum-variance corner portfolio traced out by Critical Line Algorithm
- Implemented via `pypfopt`'s own `CriticalLineAlgorithm` rather than a from-scratch port, since PyPortfolioOpt already implements the same algorithm faithfully.

In [ ]:
# ==========================================================
# CLA — Critical Line Algorithm
# Uses pypfopt.cla.CriticalLineAlgorithm, which implements CLA exactly
# min_volatility() traces the efficient frontier and returns
# the minimum-variance corner portfolio (last point on the
# frontier), replicating the paper's cla.w[-1] pattern.
# Constraints: long-only (0 <= w <= 1), sum(w) = 1.
# Note: mu must have distinct values for CLA to trace the
# frontier; the actual values do not affect min_volatility().
# ==========================================================

def getCLA(cov, **kargs):
    """Minimum variance portfolio via CriticalLineAlgorithm (Bailey & Lopez de Prado 2013)."""
    cov = pd.DataFrame(cov)
    n = cov.shape[0]
    mu = pd.Series(np.arange(1, n + 1, dtype=float), index=cov.index)
    cla = CriticalLineAlgorithm(mu, cov, weight_bounds=(0, 1))
    cla.min_volatility()
    w = cla.clean_weights()
    return np.array([w[col] for col in cov.columns])


## 4. Data Generation

- López de Prado's synthetic data-generating process (Appendix A.4) was used instead of real market data so every Monte Carlo iteration draws fresh, known-structure data
- Correlated series are built from a shared factor (one of the base series) plus noise
- Two random shocks are injected outside the estimation window to stress-test each method out-of-sample: a common shock (affecting one base series and its correlated counterpart) and a specific shock (affecting the same underlying base series that the last correlated series was built from)

In [ ]:
# ============================================================
# Data Generation
# Source: López de Prado (2016) Appendix A.4 (p.16-17)
# This is the MC version — different from A.3's generateData.
# Key additions: common random shock + specific random shock.
# ============================================================

def generateData(nObs, sLength, size0, size1, mu0, sigma0, sigma1F):
    """
    Generate correlated return series with two types of random shocks.

    sLength : estimation window (shocks placed outside it, in OOS region)
    sigma0  : base volatility
    sigma1F : factor for correlated series noise (actual noise = sigma0 * sigma1F)
    """
    #1) generate random uncorrelated data
    x = np.random.normal(mu0, sigma0, size=(nObs, size0))
    #2) create correlation between the variables
    cols = [random.randint(0, size0 - 1) for i in range(size1)]  # paper: xrange
    y = x[:, cols] + np.random.normal(0, sigma0 * sigma1F, size=(nObs, len(cols)))
    x = np.append(x, y, axis=1)
    #3) add common random shock
    point = np.random.randint(sLength, nObs - 1, size=2)
    x[np.ix_(point, [cols[0], size0])] = np.array([[-0.5, -0.5], [2, 2]])
    #4) add specific random shock
    point = np.random.randint(sLength, nObs - 1, size=2)
    x[point, cols[-1]] = np.array([-0.5, 2])
    return x, cols

## 5. Monte Carlo Experiment

- `hrpMC` ties the pieces above together into the paper's full experiment
- For each of `numIters` iterations: generate fresh synthetic data, roll forward through the sample computing in-sample weights and out-of-sample returns for all three methods (HRP/IVP/CLA) every `rebal` periods, then record each method's terminal (cumulative) OOS return
- The variance of terminal returns across iterations is the paper's headline comparison metric — lower variance means a more stable portfolio construction method

In [ ]:
# ============================================================
# Monte Carlo Experiment
# Source: López de Prado (2016) Appendix A.4 — hrpMC()
# Python 3 adaptations: print -> print(); xrange -> range;
#                       .append -> pd.concat; print pd -> print(pd)
# ============================================================

def hrpMC(numIters=10000, nObs=520, size0=5, size1=5, mu0=0, sigma0=1e-2,
          sigma1F=0.25, sLength=260, rebal=22):
    """
    Monte Carlo experiment on HRP, IVP, CLA.

    For each iteration:
      1. Generate fresh random correlated data with shocks
      2. Roll estimation window: compute weights in-sample, returns OOS
      3. Compute cumulative (terminal) return over full OOS period

    Reports: std and variance of terminal returns across all iterations.
    Reported variance matches paper Table (p.10): CLA=0.1157, IVP=0.0928, HRP=0.0671.
    """
    methods = [getIVP, getHRP, getCLA]
    stats = {i.__name__: pd.Series(dtype=float) for i in methods}
    numIter = 0
    pointers = range(sLength, nObs, rebal)

    while numIter < numIters:
        if (numIter + 1) % 1000 == 0:
            print(f'  Iteration {numIter + 1} / {numIters}')

        #1) Prepare data for one experiment
        x, cols = generateData(nObs, sLength, size0, size1, mu0, sigma0, sigma1F)
        r = {i.__name__: pd.Series(dtype=float) for i in methods}

        #2) Compute portfolios in-sample
        for pointer in pointers:
            x_ = x[pointer - sLength:pointer]
            cov_  = np.cov(x_, rowvar=0)    # rowvar=0: columns are variables
            corr_ = np.corrcoef(x_, rowvar=0)
            #3) Compute performance out-of-sample
            x_ = x[pointer:pointer + rebal]
            for func in methods:
                w_ = func(cov=cov_, corr=corr_)  # callback interface
                r_ = pd.Series(np.dot(x_, w_))
                r[func.__name__] = pd.concat([r[func.__name__], r_])  # paper: .append

        #4) Evaluate and store terminal return for this iteration
        for func in methods:
            r_ = r[func.__name__].reset_index(drop=True)
            p_ = (1 + r_).cumprod()
            stats[func.__name__].loc[numIter] = p_.iloc[-1] - 1  # terminal return

        numIter += 1

    #5) Report results
    stats = pd.DataFrame.from_dict(stats, orient='columns')
    stats.to_csv('stats.csv')  # not reloaded elsewhere in this notebook; kept for offline inspection of per-iteration terminal returns
    df0, df1 = stats.std(), stats.var()
    print(pd.concat([df0, df1, df1 / df1['getHRP'] - 1], axis=1))  # paper: print pd.concat
    return df0, df1

## 6. Sanity Check (100 iterations)

- Fast (~1 minute) run to catch implementation bugs before committing to the full 10,000-iteration run below
- Confirms the expected ordering `HRP < IVP < CLA` holds even at this small, noisier sample size

In [ ]:
# Quick sanity check: 100 iterations (~1 minute)
# Ordering should be getHRP < getIVP < getCLA before full run.

print('Running sanity check (100 iterations)...')
std_check, var_check = hrpMC(numIters=100)

print('\nVariance of terminal returns (100 iterations):')
for method in ['getCLA', 'getIVP', 'getHRP']:
    print(f'  {method}: {var_check[method]:.4f}')

ordering_ok = var_check['getHRP'] < var_check['getIVP'] < var_check['getCLA']
print(f'\nOrdering getHRP < getIVP < getCLA: {ordering_ok}')

## 7. Full Monte Carlo (10,000 iterations)

- The paper's actual experiment size
- Takes 10-20 minutes; this is the run whose numbers are compared against the paper's Table (p.10) below

In [ ]:
# Full Monte Carlo: 10,000 iterations
# Expected runtime: 10-20 minutes.

print('Running full Monte Carlo (10,000 iterations)...')
print('Progress every 1,000 iterations.\n')

std_results, var_results = hrpMC(numIters=10000)

print('\nDone.')

## 8. Results

- Prints the replicated variance of terminal returns for CLA/IVP/HRP side-by-side with the paper's own reported values, for direct comparison

In [ ]:
# ============================================================
# Results
# ============================================================

paper_var = {'getCLA': 0.1157, 'getIVP': 0.0928, 'getHRP': 0.0671}

print('=' * 52)
print('REPLICATED — variance of terminal returns')
print('=' * 52)
print(f'{"Method":<10} {"Std":>10} {"Var":>10}')
print('-' * 33)
for method in ['getCLA', 'getIVP', 'getHRP']:
    print(f'{method:<10} {std_results[method]:>10.4f} {var_results[method]:>10.4f}')

print()
print('=' * 52)
print('PAPER — López de Prado (2016) p.10')
print('=' * 52)
print(f'{"Method":<10} {"Var":>10}')
print('-' * 22)
for method, var in paper_var.items():
    print(f'{method:<10} {var:>10.4f}')

## 9. Visualisation

- Bar charts comparing the replicated vs. the paper's variance of terminal returns side-by-side, saved to `hrp_replication.png`

In [ ]:
# ============================================================
# Visualisation
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
colors  = ['#E53935', '#FB8C00', '#43A047']  # red, orange, green
methods = ['getCLA', 'getIVP', 'getHRP']
labels  = ['CLA', 'IVP', 'HRP']

vars_rep = [var_results[m] for m in methods]
bars0 = axes[0].bar(labels, vars_rep, color=colors)
axes[0].set_title('Replicated Results', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Variance of Terminal Returns')
axes[0].set_xlabel('Method')
for bar, val in zip(bars0, vars_rep):
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

vars_pap = [paper_var[m] for m in methods]
bars1 = axes[1].bar(labels, vars_pap, color=colors)
axes[1].set_title('Paper: López de Prado (2016)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Variance of Terminal Returns')
axes[1].set_xlabel('Method')
for bar, val in zip(bars1, vars_pap):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.001,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10)

plt.suptitle('HRP Monte Carlo Replication — López de Prado (2016)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('hrp_replication.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hrp_replication.png')